# Constructing the Ethylene Hamiltonian and Mapping to Qubits

## 1. Ethylene Geometry at Equilibrium

```

 H       H 
  \     /  
   C = C   
  /     \  
 H       H 

```
C=C Bond Length $\approx 1.34Å$ 

C-H Bond Length $\approx 1.087Å$

H-C-H Bond Angle $\approx 117.3^{\circ}$

Carbons are relatively positioned at $ x=0, y=0 $ and aligned along z-axis where the distance of each from the origin is equal in value.

Carbon $ |d_z| = \frac{1.34}{2}Å = 0.67Å $

At $\pm0.67$ on the z-axis where the carbon atoms are located, bonds with the hydrogen atoms form. These bonds are tilted by an angle. We calculate the displacements on $y, z$ axes.

Hydrogen $|d_y| = 1.087\sin(\frac{117.3}{2}) = 0.928Å$

Hydrogen $|d_z| = 1.087\cos(\frac{117.3}{2}) + $ Carbon $|d_z| = 0.5655Å + 0.67Å = 1.2355Å$

Meanwhile there is no displacement on the x axis because it is planar.

Now we can finally construct the PySCF driver for Ethylene at equilibrium as follows:

In [5]:
from qiskit_nature.second_q.drivers import PySCFDriver

eth_mol = PySCFDriver(
    atom="C 0 0 0.67; H 0 0.928 1.2355; H 0 -0.928 1.2355; C 0 0 -0.67; H 0 0.928 -1.2355; H 0 -0.928 -1.2355",
    basis='sto3g'
)

es_problem = eth_mol.run()

## 2. Ethylene Geometry at a Twisted $90^{\circ}$ Angle ($CH_2$ Groups Perpendicular to Eachother)

```
  H H     
  |  \    
  C - C   
  |    \  
  H     H 
```

C=C Bond Length $\approx 1.46Å$ 

C-H Bond Length $\approx 1.08Å$

H-C-H Bond Angle $\approx 121^{\circ}$

Carbon $|d_z| = 1.46/2 = 0.73Å$

But now after the twist, hydrogens are on the x-axis instead of y where the new displacement on the x-axis is equal to the one on the y-axis at equilibrium, while remaining on the z-axis as well

Hydrogen $|d_x| = 0.928$

Hydrogen $|d_z| = 1.08\cos(\frac{121}{2}) +$ Carbon $ |d_z| = 0.53Å + 0.73Å = 1.26Å$



In [6]:
eth_mol_twisted = PySCFDriver(
    atom="C 0 0 0.73; H 0.928 0 1.26; H -0.928 0 1.26; C 0 0 -0.73; H 0.928 0 -1.26; H -0.928 0 -1.26",
    basis='sto3g'
)

es_problem_twisted = eth_mol_twisted.run()

## 3. Reducing to Active Space

2 electrons, 2 orbitals expandable to 2 electrons, 4 orbitals

In [7]:
n_e = 2
n_orb = 2 # change to 4 for expansion

from qiskit_nature.second_q.transformers import ActiveSpaceTransformer

transformer = ActiveSpaceTransformer(num_electrons=n_e,num_spatial_orbitals=n_orb)

reduced_problem = transformer.transform(es_problem)
reduced_problem_twisted = transformer.transform(es_problem_twisted)

## 5. Mapping to Qubits and Getting Hamiltonian for Ethylene in Equilibrium and Twisted Geometry

In [10]:
from qiskit_nature.second_q.mappers import JordanWignerMapper

mapper = JordanWignerMapper()
h = mapper.map(reduced_problem.second_q_ops()[0]) # index of the main hamiltonian in returned tuple
h_twisted = mapper.map(reduced_problem_twisted.second_q_ops()[0])

In [ ]:
print(h)
print(h_twisted)

SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IZII', 'ZIII', 'IIZZ', 'IZIZ', 'ZIIZ', 'YYYY', 'XXYY', 'YYXX', 'XXXX', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[-0.67810865+0.j,  0.07683157+0.j, -0.07802605+0.j,  0.07683157+0.j,
 -0.07802605+0.j,  0.08418098+0.j,  0.12685108+0.j,  0.12720766+0.j,
  0.04302668+0.j,  0.04302668+0.j,  0.04302668+0.j,  0.04302668+0.j,
  0.12720766+0.j,  0.13086927+0.j,  0.08418098+0.j])
SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IZII', 'ZIII', 'IIZZ', 'IZIZ', 'ZIIZ', 'YYYY', 'XXYY', 'YYXX', 'XXXX', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[-0.65809122+0.j,  0.06206056+0.j, -0.0626729 +0.j,  0.06206056+0.j,
 -0.0626729 +0.j,  0.07940855+0.j,  0.12362153+0.j,  0.12451616+0.j,
  0.04510761+0.j,  0.04510761+0.j,  0.04510761+0.j,  0.04510761+0.j,
  0.12451616+0.j,  0.12784495+0.j,  0.07940855+0.j])
